# TP01 - Première utilisation des données

In [41]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import numpy as np

file = "../data/food.parquet"

batch_size = 70000

Chargement des données avec PyArrow

In [46]:
fichier_charge = pq.ParquetFile(file)
df_france = pd.DataFrame()
for batch in fichier_charge.iter_batches(batch_size=batch_size, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"]):
    temp_df = batch.to_pandas()
    temp_df = temp_df[temp_df["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
    df_france = pd.concat([df_france,temp_df])
print(len(df_france))

1247336


Chargement des données avec DuckDB

In [43]:
#Chargement avec duckdb
df_france = duckdb.sql("""
    SELECT "product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"
    FROM '../data/food.parquet'
    WHERE list_contains(countries_tags, 'en:france')
""").df()

Chargement des données avec Pandas

In [45]:
#Chargement avec Pandas
dt = pd.read_parquet(file, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"])
df_france = dt[dt["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
len(df_france)

1247336

Chargement des données avec Polars

In [ ]:
# Affichage des colonnes
rel = duckdb.sql("SELECT * FROM '../data/food.parquet' limit 0")
rel.columns

Récupération des éléments ayant un nutriscore renseigné

In [16]:
dt.count()
#dt[dt["nutriscore_grade"].isin(["a","b","c","d","e"])]["nutriscore_grade"].count()
dt.groupby("nutriscore_grade")["nutriscore_grade"].count()

nutriscore_grade
a                  200871
b                  159205
c                  287328
d                  346120
e                  387545
not-applicable     101213
unknown           3110742
Name: nutriscore_grade, dtype: int64

le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g )

In [ ]:
#dt = pd.read_parquet(file, columns=["product_name", "countries_tags"])

In [65]:
#dt[dt['countries_tags'] == ["en:france"]]

ValueError: ('Lengths must match to compare', (4636471,), (1,))

In [ ]:
rel = duckdb.sql("SELECT f.product_name, f.nutriments FROM '../data/food.parquet' as f limit 10").df()
rel